In [70]:
from utils import extract_parcel_mask
from nilearn import datasets, plotting
import nibabel as nib
import numpy as np
import pandas as pd

In [71]:
mask_data, mask_img = extract_parcel_mask(
    "../parcellations/neurosynth/Neurosynth_Parcellation_k200.nii.gz", 1)

In [16]:
f = nib.load("../parcellations/neurosynth/Neurosynth_Parcellation_k50.nii.gz")
f.get_fdata().max()

np.float64(49.99999998835847)

In [73]:
plotting.plot_roi(mask_img, draw_cross=False)

In [74]:
from nimare.nimads import Studyset
from nimare.utils import get_resource_path
import os

In [40]:
studyset = Studyset(
    os.path.join(get_resource_path(), "neurosynth_laird_studyset.json"),
    target="mni152_2mm",
)
studyset.annotations_df.head(5)

,id,study_id,contrast_id,Neurosynth_TFIDF__001,Neurosynth_TFIDF__01,Neurosynth_TFIDF__05,Neurosynth_TFIDF__10,Neurosynth_TFIDF__100,Neurosynth_TFIDF__11,Neurosynth_TFIDF__12,...,Neurosynth_TFIDF__yield,Neurosynth_TFIDF__yielded,Neurosynth_TFIDF__young,Neurosynth_TFIDF__young adults,Neurosynth_TFIDF__young healthy,Neurosynth_TFIDF__young older,Neurosynth_TFIDF__younger,Neurosynth_TFIDF__younger adults,Neurosynth_TFIDF__youth,Neurosynth_TFIDF__zone
0,17029760-1,17029760,1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,18760263-1,18760263,1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,19162389-1,19162389,1,0.0,0.0,0.0,0.0,0.0,0.176321,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,19603407-1,19603407,1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,20197097-1,20197097,1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
ids = studyset.get_studies_by_mask(mask_img)
decoder = discrete.NeurosynthDecoder(correction=None)
decoder.fit(studyset)
decoded_df = decoder.transform(ids=ids)
decoded_df.sort_values(by="probReverse", ascending=False).head()

INFO:nimare.decode.base:Retaining 728/3228 features.


,pForward,zForward,probForward,pReverse,zReverse,probReverse
Term,,,,,,
Neurosynth_TFIDF__human,1.601354e-07,5.240508,0.507576,0.009097,2.608387,0.820896
Neurosynth_TFIDF__provide,3.201402e-05,4.158646,0.615385,0.006265,2.733593,0.812500
Neurosynth_TFIDF__lateral,3.201402e-05,4.158646,0.615385,0.006265,2.733593,0.812500
Neurosynth_TFIDF__anatomical,3.201402e-05,4.158646,0.615385,0.006265,2.733593,0.812500
Neurosynth_TFIDF__thalamus,2.732403e-03,2.996341,0.642857,0.022534,2.281246,0.777778


In [75]:
# Run the decoder
decoder = discrete.BrainMapDecoder(correction=None)
decoder.fit(studyset)
decoded_df = decoder.transform(ids=ids)
decoded_df.sort_values(by="probReverse", ascending=False).head()

,pForward,zForward,likelihoodForward,pReverse,zReverse,probReverse
Term,,,,,,
LDA400_abstract_weight__25_approach_method_techniques,0.358178,0.918842,1.043204,0.617192,0.499833,0.036924
LDA400_abstract_weight__19_mechanisms_underlying_unknown,0.422793,0.801585,0.970669,0.756932,-0.309512,0.031784
LDA400_abstract_weight__216_frontal_inferior_gyrus,0.279590,1.081242,1.136654,0.550446,0.597092,0.031704
LDA400_abstract_weight__374_magnetic_resonance_suggest,0.496128,0.680595,0.902975,0.209142,-1.255928,0.031619
LDA400_abstract_weight__321_important_play_suggest,0.475032,0.714315,0.921676,0.396651,-0.847618,0.031614


In [76]:
# This method decodes the ROI image directly, rather than comparing subsets of the Studyset like
# the other two.
from nimare.decode import discrete

decoder = discrete.ROIAssociationDecoder(mask_img, feature_group="cogat")
decoder.fit(studyset)

# The `transform` method doesn't take any parameters.
decoded_df = decoder.transform()

decoded_df.sort_values(by="r", ascending=False)

Exception: No features identified in the input Studyset/Dataset collection!

In [89]:
from nimare.extract import fetch_neurosynth

studyset = fetch_neurosynth(
    version="7",
    source="abstract",
    vocab="LDA100",
    type="weight",
    return_type="studyset",
    target="mni152_2mm",
)[0]

INFO:nimare.extract.utils:Dataset found in /Users/amisha/.nimare/neurosynth

INFO:nimare.extract.extract:Searching for any feature files matching the following criteria: [('source-abstract', 'vocab-LDA100', 'type-weight', 'data-neurosynth', 'version-7')]


File exists and overwrite is False. Skipping.
File exists and overwrite is False. Skipping.
File exists and overwrite is False. Skipping.
File exists and overwrite is False. Skipping.
File exists and overwrite is False. Skipping.
File exists and overwrite is False. Skipping.


In [86]:
feature_columns = [
    column
    for column in studyset.annotations_df.columns
    if "__" in column
]

feature_groups = sorted({
    column.split("__", 1)[0]
    for column in feature_columns
})

print(feature_groups)
print(feature_columns[:10])

['LDA200_abstract_weight']
['LDA200_abstract_weight__0_symptoms_severity_symptom', 'LDA200_abstract_weight__1_treatment_therapy_baseline', 'LDA200_abstract_weight__2_problem_problems_arithmetic', 'LDA200_abstract_weight__3_task_switching_set', 'LDA200_abstract_weight__4_cerebellar_cerebellum_lobule', 'LDA200_abstract_weight__5_body_bodies_eba', 'LDA200_abstract_weight__6_implicit_explicit_cd', 'LDA200_abstract_weight__7_single_trial_task', 'LDA200_abstract_weight__8_cues_cue_target', 'LDA200_abstract_weight__9_semantic_word_knowledge']


In [90]:
decoder = discrete.ROIAssociationDecoder(
    mask_img,
    feature_group="LDA100_abstract_weight",  # use exact value printed above
)

decoder.fit(studyset)

decoded_df = decoder.transform()
decoded_df = decoded_df.sort_values("r", ascending=False)

print(decoded_df.head(20))

                                                           r
feature                                                     
LDA100_abstract_weight__41_memory_retrieval_epi...  0.141385
LDA100_abstract_weight__67_network_state_resting    0.138984
LDA100_abstract_weight__83_cortex_anterior_cing...  0.082289
LDA100_abstract_weight__9_imagery_mental_events     0.074454
LDA100_abstract_weight__71_reasoning_mind_mental    0.050295
LDA100_abstract_weight__76_memory_recognition_i...  0.036768
LDA100_abstract_weight__95_adults_age_older         0.034105
LDA100_abstract_weight__40_ad_disease_mci           0.033003
LDA100_abstract_weight__96_patterns_cortex_common   0.030956
LDA100_abstract_weight__30_emotional_negative_p...  0.030922
LDA100_abstract_weight__19_judgments_judgment_ppc   0.029209
LDA100_abstract_weight__94_network_control_fron...  0.028984
LDA100_abstract_weight__72_memory_encoding_hipp...  0.027468
LDA100_abstract_weight__92_social_empathy_moral     0.023419
LDA100_abstract_weight__

In [54]:
from nimare.decode import discrete

mask_data, mask_img = extract_parcel_mask(
    "../parcellations/neurosynth/Neurosynth_Parcellation_k200.nii.gz", 1)

decoder = discrete.ROIAssociationDecoder(
    mask_img,
    feature_group="Neurosynth_TFIDF",
)

studyset = Studyset(
    os.path.join(get_resource_path(), "neurosynth_laird_studyset.json"),
    target="mni152_2mm",
)

studyset.annotations_df.head(5)

decoder.fit(studyset)
decoded_df = decoder.transform()

decoded_df = decoded_df.sort_values("r", ascending=False)
print(decoded_df.head(20))

INFO:nimare.decode.base:Retaining 728/3228 features.


                                                r
feature                                          
Neurosynth_TFIDF__provide                0.699119
Neurosynth_TFIDF__position               0.677542
Neurosynth_TFIDF__medial lateral         0.677542
Neurosynth_TFIDF__medial                 0.677542
Neurosynth_TFIDF__disorders              0.677542
Neurosynth_TFIDF__lateral                0.659733
Neurosynth_TFIDF__involving              0.630488
Neurosynth_TFIDF__ofc                    0.630488
Neurosynth_TFIDF__hippocampus            0.630488
Neurosynth_TFIDF__involved cognitive     0.630488
Neurosynth_TFIDF__autonomic              0.630488
Neurosynth_TFIDF__orbitofrontal          0.630488
Neurosynth_TFIDF__cortex ofc             0.630488
Neurosynth_TFIDF__connectivity anterior  0.630488
Neurosynth_TFIDF__face                   0.630488
Neurosynth_TFIDF__portions               0.630488
Neurosynth_TFIDF__striatum               0.630488
Neurosynth_TFIDF__widespread             0.630488
